In [1]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
%matplotlib inline
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
OUT = ROOT / 'results'
OUT.mkdir(exist_ok=True)
RAW = Path(os.environ.get('ASTRO320W_DATA', str(ROOT / 'raw')))

In [ ]:
from pathlib import Path
import csv,json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy.ndimage import gaussian_filter, maximum_filter
root=RAW
out=OUT
files=sorted((root/'EXERCISE C-2/Ogma calibrated').glob('*.fits'))
# Supplementary repeatability check of an unsaturated field star.
x0,y0=221.,1237.
measure=[]
for f in files:
 d,h=fits.getdata(f,header=True);d=d.astype(float)
 x,y=x0,y0
 for _ in range(4):
  ix,iy=int(round(x)),int(round(y));sub=d[iy-9:iy+10,ix-9:ix+10]
  yy,xx=np.mgrid[iy-9:iy+10,ix-9:ix+10]
  w=np.maximum(sub-np.median(sub),0);x=float((xx*w).sum()/w.sum());y=float((yy*w).sum()/w.sum())
 ix,iy=int(round(x)),int(round(y));sub=d[iy-68:iy+69,ix-68:ix+69]
 yy,xx=np.mgrid[iy-68:iy+69,ix-68:ix+69];rr=np.hypot(xx-x,yy-y)
 ap=rr<=17;ann=(rr>=34)&(rr<=67)
 sky=np.median(sub[ann]);counts=float((sub[ap]-sky).sum());t=h['EXPTIME']
 sigma_b=1.4826*np.median(np.abs(sub[ann]-sky))
 n_ap=ap.sum();n_ann=ann.sum()
 peak=float(np.max(d[iy-8:iy+9,ix-8:ix+9]))
 measure.append([f.name,x,y,t,counts,peak,int(n_ap),int(n_ann),float(sigma_b)])
with (DATA/'field_star_pixel_measurements.csv').open('w',newline='') as f:
 w=csv.writer(f)
 w.writerow(['File','x_array_pix','y_array_pix','Exposure_s','Net_counts_DN','Peak_DN','Aperture_pixels','Annulus_pixels','Sky_sigma_DN'])
 w.writerows(measure)
print(f'Saved {len(measure)} stellar measurements. Rates and uncertainties are calculated in Excel.')
